# Chapter 8 &mdash; RE to NFA: the Thompson-Style Constructions

**Concept 2 of the Chapter 8 decomposition:** *RE to NFA: the Thompson-Style Constructions for $\varepsilon$, $a$, Concatenation, Union and Star*

One NFA fragment per RE operator, glued with $\varepsilon$ edges, with finality carefully managed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Thompson-Constructions/Concept-Thompson-Constructions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The RE-to-NFA conversion is **compositional**: one construction per operator, each
producing a fragment with **one start** and **one final** state so the next
construction can glue onto it.

* $\varepsilon$: a single state, initial and final.
* $a$: two states with an $a$-edge.
* $R_1R_2$: $\varepsilon$ from $R_1$'s final to $R_2$'s start; $R_1$'s final stops being final.
* $R_1+R_2$: a new start with $\varepsilon$ edges to both.
* $R^*$: a new start-and-final state, $\varepsilon$ into $R$, and $\varepsilon$ back from
  $R$'s final.

The $\varepsilon$ edges are what keep each rule to two lines &mdash; this is Concept 3 of
Chapter 7 paying off.

## 2. Definitions

### Jove's fragment builders, used directly

In [ ]:
from jove.Def_RE2NFA import (mk_eps_nfa, mk_symbol_nfa, mk_cat_nfa,
                             mk_plus_nfa, mk_star_nfa, ResetStNum)
def sizes(N): return "|Q|=%d Q0=%s F=%s" % (len(N["Q"]), sorted(N["Q0"]), sorted(N["F"]))

### The same thing via the parser, for comparison

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

## 3. Tests

The two primitives.

In [ ]:
ResetStNum()
E = mk_eps_nfa()
A = mk_symbol_nfa('0')
print("epsilon fragment :", sizes(E))
print("symbol  fragment :", sizes(A))
assert accepts_nfa(E, '') and accepts_nfa(A, '0')
assert E["Q0"] & E["F"], "for epsilon the start state is also final"

**Concatenation** glues with one $\varepsilon$ edge and demotes the left fragment's final state.

In [ ]:
ResetStNum()
cat = mk_cat_nfa(mk_symbol_nfa('0'), mk_symbol_nfa('1'))
print("cat fragment :", sizes(cat))
assert accepts_nfa(cat, '01')
assert not accepts_nfa(cat, '0') and not accepts_nfa(cat, '1')
print("accepts '01' only -- the left fragment's old final state is no longer final")

**Union** adds a new start with two $\varepsilon$ edges.

In [ ]:
ResetStNum()
alt = mk_plus_nfa(mk_symbol_nfa('0'), mk_symbol_nfa('1'))
print("union fragment :", sizes(alt))
assert accepts_nfa(alt, '0') and accepts_nfa(alt, '1')
assert not accepts_nfa(alt, '01')

**Star** adds a start-and-final state with $\varepsilon$ in and $\varepsilon$ back.

In [ ]:
ResetStNum()
st = mk_star_nfa(mk_symbol_nfa('0'))
print("star fragment :", sizes(st))
for s in ['', '0', '00', '000']:
    assert accepts_nfa(st, s), s
print("accepts epsilon and every 0^n -- the new state is initial AND final")

The parser composes the same fragments; the languages agree.

In [ ]:
built  = mk_star_nfa(mk_plus_nfa(mk_symbol_nfa('0'), mk_symbol_nfa('1')))
parsed = re2nfa("(0+1)*")
print("iso after minimizing? ",
      iso_dfa(min_dfa(nfa2dfa(built)), min_dfa(nfa2dfa(parsed))))
assert iso_dfa(min_dfa(nfa2dfa(built)), min_dfa(nfa2dfa(parsed)))

Fragment sizes stay **linear** in the RE length &mdash; that is the point of the construction.

In [ ]:
for r in ["0", "01", "0+1", "(0+1)*", "(0+1)*1(0+1)(0+1)"]:
    print("%-20s RE length %2d -> NFA |Q| = %2d" % (r, len(r), len(re2nfa(r)["Q"])))

## 4. Animation

A Thompson-built NFA: notice the $\varepsilon$ edges doing the gluing.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(re2nfa('(0+1)*1'), FuseEdges=True)

## 5. Exercises


1. Draw the star construction by hand. Why does the new state have to be final?
2. What goes wrong if concatenation forgets to demote the left fragment's final state?
3. How many states does the Thompson NFA for an RE of length $n$ have, roughly?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter8/Concept-Thompson-Constructions')